# Testing Lawgic Using LangChain and Groq
(because RAG in Vertex AI is paid...)

In [ ]:
# %pip install langchain langchain-community pypdf
# %pip install langsmith
# %pip install langchain-groq
# %pip install langchain langchain-text-splitters langchain-community bs4
# %pip install langchain-chroma
# %pip install sentence-transformers
# %pip install ipywidgets
# %pip install langchain-huggingface

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Note: you may need to restart the kernel to use updated packages.


In [1]:
import os
import getpass
from dotenv import load_dotenv

import json
import pandas as pd
import numpy as np

# Setup

1. Chat Model

In [19]:
from langchain_groq import ChatGroq

if not os.environ.get("GROQ_API_KEY"):
    os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your Groq API key: ")

llm = ChatGroq(
    model="llama-3.3-70b-versatile",  # or another Groq model
    temperature=0,
)

2. Embeddings Model

In [3]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")

3. Vector Store

In [4]:
from langchain_chroma import Chroma

vector_store = Chroma(
    collection_name="example_rag_langchain_collection",
    embedding_function=embeddings,
    persist_directory="../collections/chroma_langchain_db",  # Where to save data locally, remove if not necessary
)

## Testing RAG Using LangChain

### Semantic Search

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

sample_file_path = "../datasets/CUAD_v1/full_contract_pdf/Part_III/Distributor/LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGREEMENT.PDF"
loader = PyPDFLoader(sample_file_path)

docs = loader.load()

print(len(docs))


10


In [6]:
print(f"{docs[0].page_content[:200]}\n")
print(docs[0].metadata)

EXHIBIT 10.6
                              DISTRIBUTOR AGREEMENT
         THIS  DISTRIBUTOR  AGREEMENT (the  "Agreement")  is made by and between
Electric City Corp.,  a Delaware  corporation  ("Compa

{'producer': 'EVO HTML to PDF Converter 7.4', 'creator': 'PyPDF', 'creationdate': '', 'source': '../datasets/CUAD_v1/full_contract_pdf/Part_III/Distributor/LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGREEMENT.PDF', 'total_pages': 10, 'page': 0, 'page_label': '1'}


In [7]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000, chunk_overlap=200, add_start_index=True
)
all_splits = text_splitter.split_documents(docs)

print(len(all_splits))

71


In [8]:
vector_1 = embeddings.embed_query(all_splits[0].page_content)
vector_2 = embeddings.embed_query(all_splits[1].page_content)

assert len(vector_1) == len(vector_2)
print(f"Generated vectors of length {len(vector_1)}\n")
print(vector_1[:10])

Generated vectors of length 768

[0.021088002249598503, -0.0011535205412656069, 0.02359265275299549, -0.004085799679160118, -0.016694307327270508, 0.011954566463828087, 0.0715947300195694, -0.005269474349915981, 0.015024959109723568, -0.02905212715268135]


In [9]:
ids = vector_store.add_documents(documents=all_splits)

In [18]:
results = vector_store.similarity_search_with_score(
    "What happens if either the Distributor or the Company fails to comply with the terms of the agreement?"
)

for result in results:
    doc, score = result
    print(f"Score: {score}")
    print(doc)
    print("--------------------------------------------------------------------------------------------------------------------------------")
    print("\n")

Score: 0.6187871694564819
page_content='4.       DURATION AND TERMINATION
         4.1      Duration.   Unless  earlier   terminated   otherwise  provided
                  therein,  this  Agreement,  subject to the  commencement  date
                  established  in Section 1.3,  shall be effective  immediately.
                  Distributor  shall submit written  reports to the Company each
                  quarter during the first year of the Term,  commencing  ninety
                  (90) days after  execution of this  Agreement,  describing its
                  efforts,  the potential  customers it has  approached  and the
                  status of its efforts.
         4.2      Termination  for  Cause.   Either  party  may  terminate  this
                  Agreement upon 30 days
                                    Page -8-
                  prior written  notice to the other upon the  occurrence of any
                  of the following events: (A) the Distributor's failu

### RAG Agent
One formulation of a RAG application is as a simple agent with a tool that retrieves information. We can assemble a minimal RAG agent by implementing a tool that wraps our vector store:

In [20]:
from langchain.tools import tool

@tool(response_format="content_and_artifact")
def retrieve_context(query: str):
    """Retrieve information to help answer a query."""
    retrieved_docs = vector_store.similarity_search(query, k=2)
    serialized = "\n\n".join(
        (f"Source: {doc.metadata}\nContent: {doc.page_content}")
        for doc in retrieved_docs
    )
    return serialized, retrieved_docs

In [21]:
from langchain.agents import create_agent

tools = [retrieve_context]
# If desired, specify custom instructions
prompt = (
    "You have access to a tool that retrieves context from a legal document. "
    "Use the tool to help answer user queries."
)
agent = create_agent(llm, tools, system_prompt=prompt)

In [22]:
query = (
    "What happens if either the Distributor or the Company fails to comply with the terms of the agreement?\n\n"
    "Once you get the answer, search for the relevant clauses in the document and provide them."
)

for event in agent.stream(
    {"messages": [{"role": "user", "content": query}]},
    stream_mode="values",
):
    event["messages"][-1].pretty_print()

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


================================ Human Message =================================

What happens if either the Distributor or the Company fails to comply with the terms of the agreement?

Once you get the answer, search for the relevant clauses in the document and provide them.
================================== Ai Message ==================================
Tool Calls:
  retrieve_context (7850sjedz)
 Call ID: 7850sjedz
  Args:
    query: consequences of non-compliance with agreement terms by Distributor or Company
================================= Tool Message =================================
Name: retrieve_context

Source: {'source': '../datasets/CUAD_v1/full_contract_pdf/Part_III/Distributor/LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGREEMENT.PDF', 'producer': 'EVO HTML to PDF Converter 7.4', 'page_label': '7', 'page': 6, 'creationdate': '', 'total_pages': 10, 'creator': 'PyPDF', 'start_index': 4742}
Content: (C)      Company  will  timely  furnish  all of  Distributor's
             

Another common approach is a two-step chain, in which we always run a search (potentially using the raw user query) and incorporate the result as context for a single LLM query. This results in a single inference call per query, buying reduced latency at the expense of flexibility.

In this approach we no longer call the model in a loop, but instead make a single pass.

In [23]:
from langchain.agents.middleware import dynamic_prompt, ModelRequest

@dynamic_prompt
def prompt_with_context(request: ModelRequest) -> str:
    """Inject context into state messages."""
    last_query = request.state["messages"][-1].text
    retrieved_docs = vector_store.similarity_search(last_query)

    docs_content = "\n\n".join(doc.page_content for doc in retrieved_docs)

    system_message = (
        "You are a helpful assistant. Use the following context in your response:"
        f"\n\n{docs_content}"
    )

    return system_message   

agent = create_agent(llm, tools=[], middleware=[prompt_with_context])

In [24]:
query = "What happens if either the Distributor or the Company fails to comply with the terms of the agreement?"
for step in agent.stream(
    {"messages": [{"role": "user", "content": query}]},
    stream_mode="values",
):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

What happens if either the Distributor or the Company fails to comply with the terms of the agreement?
================================== Ai Message ==================================

According to the agreement, if either the Distributor or the Company fails to comply with the terms of the agreement, the other party may terminate the agreement upon 30 days' prior written notice. This is stated in Section 4.2, "Termination for Cause".

Specifically, the agreement states that either party may terminate the agreement if the other party fails to make payments, or if there are other events that trigger termination, such as the Distributor's failure to meet its obligations.

Additionally, if the Company terminates the agreement without cause and for reasons other than the Distributor's failure to meet its minimum expectations, the Company shall repurchase from the Distributor any products in the Distributor's 